In [52]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error, root_mean_squared_error, r2_score, mean_squared_error
import matplotlib.pyplot as plt

tabela = pd.read_csv("base_vendas_atividade2_final1.csv", sep=";", encoding="latin-1")
vistoria = pd.read_csv("base_vistorias_atividade2_final.csv", sep=";", encoding="latin-1")
destinacoes = pd.read_csv("base_destinacoes_atividade2_final.csv", sep=";", encoding="latin-1")

tabela = tabela.merge(
    vistoria[["CD_IMOVEL_URBANO", "SIM_SITIMO_DS"]],
    how="left",
    left_on="CD_IMOVEL",
    right_on="CD_IMOVEL_URBANO"
)

tabela = tabela.merge(
    destinacoes[["COD_DESTINACAO_IMOVEL", "TIPO_USO_DESTINACAO", "RESIDENCIAL", "COMERCIAL", "INDUSTRIAL", "INSTITUCIONAL"]],
    how="left",
    left_on="COD_DESTINACAO_IMOVEL",
    right_on="COD_DESTINACAO_IMOVEL"
)       


# # remove a coluna duplicada da chave
tabela = tabela.drop(columns=["CD_IMOVEL_URBANO"])
tabela = tabela.rename(columns={"SIM_SITIMO_DS": "SITUACAO_VISTORIA"})
# tabela["SITUACAO_VISTORIA"] = tabela["SITUACAO_VISTORIA"].fillna("SEM_VISTORIA")


# # colunas que definem a duplicidade: mesmo ano, mesmo edital e mesmo item do edital (item de edital com imoveis agrupados)
cols = ["ANO_VENDA", "NR_EDITAL", "ITEM_EDITAL"]
# # máscara: True para linhas que aparecem em duplicidade (em qualquer posição do grupo)
mask_dup = tabela.duplicated(subset=cols, keep=False)
# # remove TODAS as linhas duplicadas desses grupos
tabela_sem_dups = tabela.loc[~mask_dup].copy()

# # Foram removidos 145 itens que constavam como items agrupados, de um total de 2982, restando 2837 registros 
# print("Linhas originais:", len(tabela))
# print("Linhas removidas:", mask_dup.sum())
# print("Linhas finais:", len(tabela_sem_dups))

tabela = tabela_sem_dups
tabela["NR_EDITAL"] = tabela["ANO_VENDA"].astype(str) + "-" + tabela["NR_EDITAL"].astype(str)

def br_to_float(s):
    """
    Converte número no formato BR para float:
    - remove separador de milhar (.)
    - troca decimal (,) por (.)
    """
    if pd.isna(s):
        return np.nan
    s = str(s).strip()
    if s == "":
        return np.nan
    s = s.replace(".", "")      # remove milhares
    s = s.replace(",", ".")     # troca decimal
    return pd.to_numeric(s, errors="coerce")

cols = ["VALOR_VENDA", "AREA_MAX_CONSTR", "AREA_BASE", "AREA", "VALOR_LAUDO"]  # ajuste
for c in cols:
    if c in tabela.columns:
        tabela[c] = tabela[c].apply(br_to_float)

# #criacao da coluna AGIO ABSOLUTO , esta coluna deve ser retirada do treino
tabela["AGIO_ABSOLUTO"] = tabela["VALOR_VENDA"] - tabela["VALOR_LAUDO"]
tabela["AGIO_PERCENTUAL"] = ((tabela["VALOR_VENDA"] - tabela["VALOR_LAUDO"]) / tabela["VALOR_LAUDO"]) * 100

#retirando os apartamentos da base (14 registros) que tem área máxima de construção igual a zero e área base igual a zero, ou seja, não tem área construída, o que é um erro de cadastro
# tabela[~(tabela["AREA_MAX_CONSTR"] == 0) & (tabela["AREA_BASE"] == 0)]

tabela = tabela[~((tabela["AREA_MAX_CONSTR"] == 0) & (tabela["AREA_BASE"] == 0))]
# tabela = tabela[~((tabela["CD_IMOVEL"] == 158423) | (tabela["CD_IMOVEL"] == 197972))]



In [53]:
df = tabela.copy()
df.columns = df.columns.str.strip()

TARGET = "VALOR_VENDA"

# Remover colunas que você NÃO quer usar
# (vazamento/colinearidade/dado futuro)
DROP_COLS = ["AGIO_ABSOLUTO", "AGIO_PERCENTUAL", "VALOR_LAUDO", "QTD_OFERTAS"]
DROP_COLS = [c for c in DROP_COLS if c in df.columns]

# Percentual 0–100 -> 0–1
if "PERCENTUAL_ENTRADA" in df.columns:
    df["PERCENTUAL_ENTRADA"] = pd.to_numeric(df["PERCENTUAL_ENTRADA"], errors="coerce") / 100.0


# Separar X e y
y = df[TARGET].copy()
X = df.drop(columns=[TARGET] + DROP_COLS)

# DS_CIDADE;DS_SETOR;COD_DESTINACAO_IMOVEL;TIPO_USO_DESTINACAO;
# Definir colunas numéricas e categóricas explicitamente
num_cols = [
    "AREA_MAX_CONSTR", "AREA_BASE", "AREA", "PERCENTUAL_ENTRADA", "RESIDENCIAL", "COMERCIAL", "INDUSTRIAL", "INSTITUCIONAL"]

num_cols = [c for c in num_cols if c in X.columns]

cat_cols = ["DS_CIDADE", "DS_SETOR", "SITUACAO_VISTORIA", "COD_DESTINACAO_IMOVEL", "ANO_VENDA", "NR_EDITAL", "TIPO_USO_DESTINACAO", "ITEM_EDITAL"]
cat_cols = [c for c in cat_cols if c in X.columns]

# Tipos (evita problemas de mixed types)
for c in cat_cols:
    X[c] = X[c].astype("string")
for c in num_cols:
    X[c] = pd.to_numeric(X[c], errors="coerce")

print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)
print("Shape X:", X.shape, "| Shape y:", y.shape)

Numéricas: ['AREA_MAX_CONSTR', 'AREA_BASE', 'AREA', 'PERCENTUAL_ENTRADA', 'RESIDENCIAL', 'COMERCIAL', 'INDUSTRIAL', 'INSTITUCIONAL']
Categóricas: ['DS_CIDADE', 'DS_SETOR', 'SITUACAO_VISTORIA', 'COD_DESTINACAO_IMOVEL', 'ANO_VENDA', 'NR_EDITAL', 'TIPO_USO_DESTINACAO', 'ITEM_EDITAL']
Shape X: (2823, 17) | Shape y: (2823,)


In [54]:
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# print("numero de linhas e colunas dos dados de treino:", X_treino.shape)
# print("numero de linhas e colunas dos dados de treino:", X_teste.shape)

escalonador = StandardScaler()
categorizador = OneHotEncoder(handle_unknown="ignore")
imputador_numero = SimpleImputer(strategy="median")
imputador_categorico = SimpleImputer(strategy="most_frequent")

# etapas_numericas
etapas_numericas = Pipeline(
    [
        ("imputer", imputador_numero),
        ("scaler", escalonador)
    ]
)


#etapas categoricas
etapas_categoricas = Pipeline(
    [
        ("imputer", imputador_categorico),
        ("encoder", categorizador)
    ]
)


preprocessador = ColumnTransformer(
    transformers=[
        ('numericas', etapas_numericas, num_cols),
        ('categoricas', etapas_categoricas, cat_cols)
    ]
)

x_treino_transformado = preprocessador.fit_transform(X_treino)
x_teste_transformado = preprocessador.transform(X_teste)

modelo = DecisionTreeRegressor(random_state=42)

param_grid = {
    "max_depth": [2,3,5,7], # profundidade maxima da arvore
    "min_samples_split": [2,5,10,15,20,25] # numero minimo de amostras para dividir um nó
}

grid_search = GridSearchCV(
    estimator= modelo, # modelo a ser otimizado
    param_grid= param_grid, # grade de hiperparamentros
    cv=5, # 5-fold cross validation
    scoring="neg_root_mean_squared_error" # metricas de comparacao RMSE
)

grid_search.fit(x_treino_transformado, y_treino)
melhor_modelo_arvore = grid_search.best_estimator_

y_pred_treino_arvore = melhor_modelo_arvore.predict(x_treino_transformado)


mse_treino_arvore = mean_absolute_error(y_treino, y_pred_treino_arvore)
rmse_treino_arvore = root_mean_squared_error(y_treino, y_pred_treino_arvore)
mae_treino_arvore = mean_absolute_error(y_treino, y_pred_treino_arvore)
r2_treino_arvore = r2_score(y_treino, y_pred_treino_arvore)
mape_treino_arvore = mean_absolute_percentage_error(y_treino , y_pred_treino_arvore)


print("Melhores parametros encontrados:", grid_search.best_params_)
print("R2 no treino: ", r2_treino_arvore)   
print("MSE no treino: ", mse_treino_arvore)
print("RMSE no treino: ", rmse_treino_arvore)
print("MAE no treino: ", mae_treino_arvore)
print("MAPE no treino (%): ", mape_treino_arvore)

modelo_knn = KNeighborsRegressor()

param_grid_knn = {
    "n_neighbors": [3,5,7,9,11], # numero de vizinhos
    "p": [1,2] # distancia (1=manhattan, 2=euclediana)
}

grid_search_knn = GridSearchCV(
    estimator= modelo_knn, # modelo a ser otimizado
    param_grid= param_grid_knn, # grade de hiperparamentros
    cv=5, # 5-fold cross validation
    scoring="neg_root_mean_squared_error" # metricas de comparacao RMSE
)

grid_search_knn.fit(x_treino_transformado, y_treino)
melhor_modelo_knn = grid_search_knn.best_estimator_

y_pred_treino_knn = melhor_modelo_knn.predict(x_treino_transformado)

mse_treino_knn = mean_squared_error(y_treino, y_pred_treino_knn)
rmse_treino_knn = root_mean_squared_error(y_treino, y_pred_treino_knn)
mae_treino_knn = mean_absolute_error(y_treino, y_pred_treino_knn)
r2_treino_knn = r2_score(y_treino, y_pred_treino_knn)
mape_treino_knn = mean_absolute_percentage_error(y_treino, y_pred_treino_knn)

print("----------------------------------------------------------")
print("Melhores parametros encontrados:", grid_search_knn.best_params_)
print("R2 no treino: ", r2_treino_knn)
print("MSE no treino: ", mse_treino_knn) 
print("RMSE no treino: ", rmse_treino_knn)
print("MAE no treino: ", mae_treino_knn)
print("MAPE no treino (%): ", mape_treino_knn)



Melhores parametros encontrados: {'max_depth': 7, 'min_samples_split': 10}
R2 no treino:  0.9942057597588679
MSE no treino:  248479.50737567423
RMSE no treino:  694090.6661847339
MAE no treino:  248479.50737567423
MAPE no treino (%):  0.382720619078287
----------------------------------------------------------
Melhores parametros encontrados: {'n_neighbors': 3, 'p': 1}
R2 no treino:  0.6462422787367853
MSE no treino:  29413170351176.32
RMSE no treino:  5423391.038010842
MAE no treino:  345303.98024062585
MAPE no treino (%):  0.25613015371844433


In [55]:
y_pred_teste_arvore = melhor_modelo_arvore.predict(x_teste_transformado)

# metricas de avaliacao
mse_teste_arvore = mean_squared_error(y_teste, y_pred_teste_arvore)
rmse_teste_arvore = root_mean_squared_error(y_teste, y_pred_teste_arvore)
mae_teste_arvore = mean_absolute_error(y_teste, y_pred_teste_arvore)
r2_teste_arvore = r2_score(y_teste, y_pred_teste_arvore)
mape_teste_arvore = mean_absolute_percentage_error(y_teste, y_pred_teste_arvore)

print("Avaliacao do modelo de arvore de decisao no teste:")
print("R2 no teste: ",  r2_teste_arvore )
print("MSE no teste: ", mse_teste_arvore)
print("RMSE no teste: ", rmse_teste_arvore)
print("MAE no teste: ", mae_teste_arvore)
print("MAPE no teste (%): ", mape_teste_arvore)
print("----------------------------------------------------------")

y_pred_teste_knn = melhor_modelo_knn.predict(x_teste_transformado)
mse_teste_knn = mean_squared_error(y_teste, y_pred_teste_knn)
rmse_teste_knn = root_mean_squared_error(y_teste, y_pred_teste_knn)
mae_teste_knn = mean_absolute_error(y_teste, y_pred_teste_knn)
r2_teste_knn = r2_score(y_teste, y_pred_teste_knn)
mape_teste_knn = mean_absolute_percentage_error(y_teste, y_pred_teste_knn)

print("Avalicao do modelo KNN no teste:")
print("R2 no teste: ",  r2_teste_knn )
print("MSE no teste: ", mse_teste_knn)
print("RMSE no teste: ", rmse_teste_knn)
print("MAE no teste: ", mae_teste_knn)
print("MAPE no teste (%): ", mape_teste_knn)



Avaliacao do modelo de arvore de decisao no teste:
R2 no teste:  0.7026613198861671
MSE no teste:  3907657421857.4336
RMSE no teste:  1976779.558235423
MAE no teste:  513478.68942802184
MAPE no teste (%):  0.47503877584915044
----------------------------------------------------------
Avalicao do modelo KNN no teste:
R2 no teste:  0.8912130119865254
MSE no teste:  1429690482750.5925
RMSE no teste:  1195696.65164313
MAE no teste:  392774.37472566374
MAPE no teste (%):  0.4387086649048806


In [62]:
modelo_et = ExtraTreesRegressor(
    random_state=42,
    n_estimators=800,
    max_depth=25,
    max_features=0.7,
    min_samples_leaf=2,
    min_samples_split=5,
    n_jobs=-1
)

modelo_et.fit(x_treino_transformado, y_treino)

y_pred_treino_et = modelo_et.predict(x_treino_transformado)

mse_treino_et = mean_squared_error(y_treino, y_pred_treino_et)
rmse_treino_et = root_mean_squared_error(y_treino, y_pred_treino_et)
mae_treino_et = mean_absolute_error(y_treino, y_pred_treino_et)
r2_treino_et = r2_score(y_treino, y_pred_treino_et)
mape_treino_et = mean_absolute_percentage_error(y_treino, y_pred_treino_et)

print("Parametros usados:", modelo_et.get_params())
print("R2 no treino:",  r2_treino_et)
print("MSE no treino:",  mse_treino_et)
print("RMSE no treino:",  rmse_treino_et)
print("MAE no treino:",  mae_treino_et)
print("MAPE no treino (%):",  mape_treino_et)

y_pred_teste_et = modelo_et.predict(x_teste_transformado)
mse_teste_et = mean_squared_error(y_teste, y_pred_teste_et)
rmse_teste_et = root_mean_squared_error(y_teste, y_pred_teste_et)
mae_teste_et = mean_absolute_error(y_teste, y_pred_teste_et)
r2_teste_et = r2_score(y_teste, y_pred_teste_et)
mape_teste_et = mean_absolute_percentage_error(y_teste, y_pred_teste_et)

print("----------------------------------------------------------")
print("Avaliacao do modelo Extra Trees no teste:")
print("R2 no teste: ",  r2_teste_et)
print("MSE no teste: ", mse_teste_et)
print("RMSE no teste: ", rmse_teste_et)
print("MAE no teste: ", mae_teste_et)
print("MAPE no teste (%): ", mape_teste_et)

Parametros usados: {'bootstrap': False, 'ccp_alpha': 0.0, 'criterion': 'squared_error', 'max_depth': 25, 'max_features': 0.7, 'max_leaf_nodes': None, 'max_samples': None, 'min_impurity_decrease': 0.0, 'min_samples_leaf': 2, 'min_samples_split': 5, 'min_weight_fraction_leaf': 0.0, 'monotonic_cst': None, 'n_estimators': 800, 'n_jobs': -1, 'oob_score': False, 'random_state': 42, 'verbose': 0, 'warm_start': False}
R2 no treino: 0.7058485044270963
MSE no treino: 24457213308148.832
RMSE no treino: 4945423.471063811
MAE no treino: 277797.9161058553
MAPE no treino (%): 0.1176595749026604
----------------------------------------------------------
Avaliacao do modelo Extra Trees no teste:
R2 no teste:  0.23622964765443388
MSE no teste:  10037553421556.906
RMSE no teste:  3168209.8133736197
MAE no teste:  396768.9051630449
MAPE no teste (%):  0.22430274851375367
